# v4b analysis (no z-loss): the "self" diagnostic (continual vs Hindi-only, both from scratch)
For every saved checkpoint: record MLP neuron activations on fixed inputs, correlate every pair of neurons,
link pairs with |r| >= tau, and call the largest connected group the **self** (the rest = **task**).
At every transition between consecutive checkpoints: match neurons (Hungarian), measure each neuron's change,
and compare self vs task vs a random group of the same size. Also tracked: whether the self keeps the **same members**.

* Phase 0 is the random model, so the transition 0 → 1 is skipped (it is not a language switch).
* Both runs are cut to the same last phase (`MATCH_PHASES`), so the comparison is fair while the continual run is still training.
* Runs on GPU 1 and only reads checkpoints, so it can run while training continues on GPU 0.

In [1]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # run this cell FIRST in a fresh kernel
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from common_v4 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

device: cuda | NVIDIA GeForce RTX 3090


In [2]:
pwd

'/workspace/shreshth'

In [3]:
ls

 Untitled-1.ipynb
 Untitled.ipynb
 __pycache__/
 analysis/
'analysis (3).ipynb'
 analysis_v4.ipynb
'analysis_v4b (2).ipynb'
'analysis_v4b (3).ipynb'
 analysis_v4b.ipynb
 arabic.ipynb
 arabic_english_hindi_chinese_tokenizer_train.ipynb
 common.py
 common_v4.py
 data/
 encoder_decoder_hindi.ipynb
 runs/
 tokenizer/
 tokenizer_train.ipynb
 tokenizer_v4/
 train_continual_gpu0.ipynb
 train_controls_gpu1.ipynb
 v4b_continual_gpu0.ipynb
 v4b_control_hi_gpu1.ipynb


In [4]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from IPython.display import display
torch.backends.cuda.matmul.allow_tf32 = False      # full precision for correlations

RUNS = {"continual": "v4b_continual_s0", "only_hi": "v4b_only_hi_s0"}
TAUS = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]  # all computed in one pass
TAU_PLOT = 0.3                         # used for plots; pick it from the tau sweep (section 4)
MAX_T = 24000                          # token positions per trace (fixed random subset)
N_RANDOM = 20                          # random size-matched groups per transition
MATCH_PHASES = True                    # cut every run to the last phase all runs have reached
MAX_PHASE = None                       # or set e.g. 6 to analyse phases 0-6 only (quick test)
OUT = P("analysis", "v4b"); os.makedirs(OUT, exist_ok=True)

def ckpts_of(run):
    return sorted(glob.glob(P("runs", run, "ckpt", "p*.pt")))
def phase_of(path):
    return int(os.path.basename(path)[1:4])

avail = {k: ckpts_of(v) for k, v in RUNS.items()}
for k, v in avail.items():
    print(f"{k:10s}: {len(v)} checkpoints, last phase {phase_of(v[-1]) if v else None}")
last = min(phase_of(v[-1]) for v in avail.values() if v) if MATCH_PHASES else None
if MAX_PHASE is not None:
    last = MAX_PHASE if last is None else min(last, MAX_PHASE)
print("analysing phases 0 ..", last if last is not None else "all")

continual : 19 checkpoints, last phase 18
only_hi   : 19 checkpoints, last phase 18
analysing phases 0 .. 18


## 1. Sanity check: phase 1 should be (nearly) the same model in both runs

In [5]:
p1 = {k: [p for p in v if phase_of(p) == 1] for k, v in avail.items()}
if all(p1.values()):
    a = torch.load(p1["continual"][0], map_location="cpu")["model"]
    b = torch.load(p1["only_hi"][0], map_location="cpu")["model"]
    rel = {k: ((a[k].float() - b[k].float()).norm() / (a[k].float().norm() + 1e-12)).item() for k in a}
    print(f"phase-1 weights, relative difference: median {np.median(list(rel.values())):.2e}, max {max(rel.values()):.2e}")
    print("(small but non-zero is expected: GPU arithmetic is not perfectly deterministic)")

phase-1 weights, relative difference: median 3.77e-02, max 1.01e+00
(small but non-zero is expected: GPU arithmetic is not perfectly deterministic)


## 2. Recording activations on fixed inputs

In [6]:
pwd

'/workspace/shreshth'

In [7]:
ls

 Untitled-1.ipynb
 Untitled.ipynb
 __pycache__/
 analysis/
'analysis (3).ipynb'
 analysis_v4.ipynb
'analysis_v4b (2).ipynb'
'analysis_v4b (3).ipynb'
 analysis_v4b.ipynb
 arabic.ipynb
 arabic_english_hindi_chinese_tokenizer_train.ipynb
 common.py
 common_v4.py
 data/
 encoder_decoder_hindi.ipynb
 runs/
 tokenizer/
 tokenizer_train.ipynb
 tokenizer_v4/
 train_continual_gpu0.ipynb
 train_controls_gpu1.ipynb
 v4b_continual_gpu0.ipynb
 v4b_control_hi_gpu1.ipynb


In [8]:
ls data

en-hi/  en-yo/  en-zh/  flores/  spm.model  spm.vocab


In [1]:
DEV_EN = read_flores("dev", "en")
SRC_IDS = [sp.encode(s)[:MAX_TOK - 1] + [EOS] for s in DEV_EN]
TGT_IN = {l: [[LANG_TAG[l]] + sp.encode(r)[:MAX_TOK - 1] for r in read_flores("dev", l)] for l in LANGS}

@torch.no_grad()
def record(model, bs=64):
    """MLP (GELU-output) activations at every real token.
    enc{j}: English tokens (same input for every checkpoint).
    dec{j}: target tokens of hi/zh/ar, teacher-forced; dec_lang = language of each position."""
    model.eval()
    cur = {}
    hooks = [b.mlp.act.register_forward_hook(lambda m, i, o, k=f"enc{j}": cur.__setitem__(k, o))
             for j, b in enumerate(model.enc_blocks)]
    hooks += [b.mlp.act.register_forward_hook(lambda m, i, o, k=f"dec{j}": cur.__setitem__(k, o))
              for j, b in enumerate(model.dec_blocks)]
    names = [f"enc{j}" for j in range(len(model.enc_blocks))] + [f"dec{j}" for j in range(len(model.dec_blocks))]
    acts, dec_lang = {k: [] for k in names}, []
    for s in range(0, len(SRC_IDS), bs):
        src = pad_batch(SRC_IDS[s:s + bs])
        mem, src_mask = model.encode(src)
        keep = (src != PAD) & (src != EOS)
        for k in names:
            if k.startswith("enc"):
                acts[k].append(cur[k][keep].half())
        for li, l in enumerate(LANGS):
            tgt = pad_batch(TGT_IN[l][s:s + bs])
            model.decode(tgt, mem, src_mask)
            keep = tgt != PAD
            keep[:, 0] = False                              # skip the language-tag position
            for k in names:
                if k.startswith("dec"):
                    acts[k].append(cur[k][keep].half())
            dec_lang.append(torch.full((int(keep.sum()),), li, device=DEVICE))
    for h in hooks:
        h.remove()
    return {k: torch.cat(v) for k, v in acts.items()}, torch.cat(dec_lang)

def subsample_idx(n, k, seed=0):
    if n <= k:
        return torch.arange(n, device=DEVICE)
    return torch.tensor(np.sort(np.random.default_rng(seed).choice(n, k, replace=False)), device=DEVICE)

def dec_subsample(dec_lang, k, seed=1):
    """Equal number of decoder positions from each language."""
    g, lang_np, out = np.random.default_rng(seed), dec_lang.cpu().numpy(), []
    for li in range(len(LANGS)):
        pos = np.nonzero(lang_np == li)[0]
        out.append(np.sort(g.choice(pos, min(len(pos), k // len(LANGS)), replace=False)))
    return torch.tensor(np.concatenate(out), device=DEVICE)

NameError: name 'read_flores' is not defined

## 3. The diagnostic: traces → correlations → blocks → matching → change

In [ ]:
def traces(X, groups=None):
    """X: (T, d). Remove each language's mean per neuron (decoder), then z-score each neuron."""
    X = X.clone()
    if groups is not None:
        for g in groups.unique():
            m = groups == g
            X[m] -= X[m].mean(0, keepdim=True)
    mu, sd = X.mean(0, keepdim=True), X.std(0, unbiased=False, keepdim=True)
    alive = sd[0] > 1e-6
    Z = (X - mu) / torch.where(sd > 0, sd, torch.ones_like(sd))
    Z[:, ~alive] = 0
    return Z, alive

def largest_block(R, alive, tau):
    """Link alive neurons with |r| >= tau; largest connected group = self."""
    if int(alive.sum()) == 0:
        return torch.zeros_like(alive), 0
    A = (R.abs() >= tau) & alive[:, None] & alive[None, :]
    A.fill_diagonal_(False)
    _, lab = connected_components(csr_matrix(A.cpu().numpy()), directed=False)
    lab = np.where(alive.cpu().numpy(), lab, -1)
    sizes = np.bincount(lab[lab >= 0])
    return torch.tensor(lab == sizes.argmax(), device=DEVICE), int(sizes.max())

def transition(Za, Ra, alive_a, Zb, Rb, alive_b):
    """Hungarian matching a -> b, then per-neuron % change = 100 x (1 - mean(activation sim, connectivity sim))."""
    S = (Za.T @ Zb) / Za.shape[0]
    _, perm = linear_sum_assignment(-S.double().cpu().numpy())
    perm = torch.tensor(perm, device=DEVICE)
    act = S[torch.arange(len(perm), device=DEVICE), perm]
    A_ = Ra.clone(); A_.fill_diagonal_(0)
    B_ = Rb[perm][:, perm].clone(); B_.fill_diagonal_(0)
    conn = (A_ * B_).sum(1) / (A_.norm(dim=1) * B_.norm(dim=1) + 1e-12)
    change = 100 * (1 - (act + conn) / 2)
    valid = alive_a & alive_b[perm]
    return perm, change, valid

def analyze_run(name, run):
    paths = [p for p in avail[name] if last is None or phase_of(p) <= last]
    print(f"\n=== {name} ({run}): {len(paths)} checkpoints ===")
    block_rows, trans_rows, quad_rows, idx, prev, keep_R = [], [], [], {}, None, {}
    for path in paths:
        t0 = time.time()
        model, ck = load_model(path)
        phase, lang = ck["phase"], ck["lang"]
        acts, dec_lang = record(model)
        if not idx:
            idx["enc"] = subsample_idx(acts["enc0"].shape[0], MAX_T)
            idx["dec"] = dec_subsample(dec_lang, MAX_T)
        layers = {}
        for lname, X in acts.items():
            kind = lname[:3]
            Z, alive = traces(X[idx[kind]].float(), dec_lang[idx["dec"]] if kind == "dec" else None)
            R = Z.T @ Z / Z.shape[0]
            selfm = {}
            for tau in TAUS:
                m, size = largest_block(R, alive, tau)
                selfm[tau] = m
                block_rows.append({"run": name, "phase": phase, "lang": lang, "layer": lname, "tau": tau,
                                   "self_size": size, "n_alive": int(alive.sum()),
                                   "self_frac": size / max(int(alive.sum()), 1)})
            layers[lname] = (Z, R, alive, selfm)
            keep_R[(phase, lname)] = (R.cpu().numpy().astype(np.float16), alive.cpu().numpy())
        del acts
        if prev is not None and prev["phase"] >= 1:          # skip random model -> phase 1
            gen = torch.Generator(device=DEVICE).manual_seed(int(phase))
            for lname, (Zb, Rb, alive_b, selfm_b) in layers.items():
                Za, Ra, alive_a, selfm_a = prev["layers"][lname]
                perm, change, valid = transition(Za, Ra, alive_a, Zb, Rb, alive_b)
                vidx = valid.nonzero().squeeze(1)
                for tau in TAUS:
                    in_self = valid & selfm_b[tau][perm]       # target-side membership, as in the paper
                    in_task = valid & ~selfm_b[tau][perm]
                    ns = int(in_self.sum())
                    sc = change[in_self].mean().item() if ns else float("nan")
                    tc = change[in_task].mean().item() if in_task.any() else float("nan")
                    rc = float("nan")
                    if 0 < ns < len(vidx):
                        rc = float(np.mean([change[vidx[torch.randperm(len(vidx), generator=gen,
                                            device=DEVICE)[:ns]]].mean().item() for _ in range(N_RANDOM)]))
                    sa, sb = selfm_a[tau], selfm_b[tau][perm]  # self before, and partners' self membership after
                    union = int((sa | sb).sum())
                    overlap = int((sa & sb).sum()) / union if union else float("nan")
                    trans_rows.append({"run": name, "from_phase": prev["phase"], "to_phase": phase,
                                       "from_lang": prev["lang"], "to_lang": lang, "layer": lname, "tau": tau,
                                       "n_self": ns, "self_change": sc, "task_change": tc, "random_change": rc,
                                       "gap": tc - sc, "gap_vs_random": rc - sc, "self_overlap": overlap})
                    for fa in (True, False):                       # 2x2: came from self/task -> is now in self/task
                        for fb in (True, False):
                            m = valid & (sa == fa) & (sb == fb)
                            quad_rows.append({"run": name, "from_phase": prev["phase"], "to_phase": phase, "layer": lname, "tau": tau,
                                              "came_from": "self" if fa else "task", "now_in": "self" if fb else "task",
                                              "n": int(m.sum()),
                                              "persistence": (1 - change[m] / 100).mean().item() if m.any() else float("nan")})
        prev = {"phase": phase, "lang": lang, "layers": layers}
        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        print(f"  phase {phase:2d} ({lang}) done in {time.time() - t0:.0f}s")
    B, T, Q = pd.DataFrame(block_rows), pd.DataFrame(trans_rows), pd.DataFrame(quad_rows)
    Q.to_csv(f"{OUT}/{name}_quad.csv", index=False)
    B.to_csv(f"{OUT}/{name}_blocks.csv", index=False)
    T.to_csv(f"{OUT}/{name}_transitions.csv", index=False)
    return {"blocks": B, "trans": T, "quad": Q, "R": keep_R, "last_phase": prev["phase"] if prev else None}

In [ ]:
results = {name: analyze_run(name, run) for name, run in RUNS.items() if avail[name]}
LAYERS = sorted({l for r in results.values() for (_, l) in r["R"]}, key=lambda s: (s[:3] != "enc", s))
print("layers:", LAYERS)

## 4. Tau sweep: size of the self (fraction of alive neurons)
Pick `TAU_PLOT` where the self is a real part of the layer, but not the whole layer.

In [ ]:
for name, r in results.items():
    B = r["blocks"]
    for ph, label in [(1, "after phase 1 (Hindi only, no switch yet)"), (r["last_phase"], "latest phase")]:
        tab = B[B.phase == ph].pivot(index="tau", columns="layer", values="self_frac")[LAYERS]
        print(f"{name}, phase {ph}: {label}")
        display(tab.round(2))

## 5. Training curves (script-restricted chrF and val loss after each phase)

In [ ]:
def read_phases(run):
    rows = {}
    for l in open(P("runs", run, "phases.jsonl")):
        r = json.loads(l); rows[r["phase"]] = r
    return [rows[k] for k in sorted(rows) if last is None or k <= last]

COL = {"hi": "C1", "zh": "C3", "ar": "C2", "init": "C7"}
fig, axes = plt.subplots(len(results), 2, figsize=(14, 3.3 * len(results)), squeeze=False)
for row, name in enumerate(results):
    ph = read_phases(RUNS[name])
    x = [r["phase"] for r in ph]
    for l in LANGS:
        axes[row, 0].plot(x, [r["chrf_masked"][l] for r in ph], marker="o", color=COL[l], label=l)
        axes[row, 1].plot(x, [r["val_masked"][l] for r in ph], marker="o", color=COL[l], label=l)
    axes[row, 0].set_title(f"{name}: chrF"); axes[row, 1].set_title(f"{name}: val loss")
    for ax in axes[row]:
        ax.set_xlabel("phase")
        for r in ph:
            ax.axvspan(r["phase"] - 0.5, r["phase"] + 0.5, alpha=0.07, color=COL.get(r["lang"], "C7"))
    axes[row, 0].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/fig_training_curves.png", dpi=120); plt.show()

for name in results:
    tab = pd.DataFrame([{"phase": r["phase"], "trained": r["lang"], "steps": r["steps"], "stop": r["reason"],
                         **{f"chrF {l}": r["chrf_masked"][l] for l in LANGS},
                         **{f"val {l}": r["val_masked"][l] for l in LANGS}} for r in read_phases(RUNS[name])])
    print(name); display(tab)
    tab.to_csv(f"{OUT}/{name}_phases.csv", index=False)

## 6. The self block, highlighted (latest phase, self neurons first, gold box)

In [ ]:
def self_mask_np(R, alive, tau):
    A = (np.abs(R) >= tau) & alive[:, None] & alive[None, :]
    np.fill_diagonal(A, False)
    _, lab = connected_components(csr_matrix(A), directed=False)
    lab = np.where(alive, lab, -1)
    return lab == np.bincount(lab[lab >= 0]).argmax()

def plot_self_blocks(phase_for, title):
    fig, axes = plt.subplots(len(results), len(LAYERS), figsize=(2.6 * len(LAYERS), 3.0 * len(results)), squeeze=False)
    for row, (name, r) in enumerate(results.items()):
        ph = phase_for(r)
        for col, l in enumerate(LAYERS):
            R, alive = r["R"][(ph, l)]
            R = R.astype(np.float32)
            m = self_mask_np(R, alive, TAU_PLOT)
            s_idx = np.where(m)[0]
            s_idx = s_idx[np.argsort(-np.abs(R[np.ix_(s_idx, s_idx)]).mean(1))]
            order = np.concatenate([s_idx, np.where(alive & ~m)[0]])
            ax = axes[row, col]
            ax.imshow(np.abs(R[np.ix_(order, order)]), vmin=0, vmax=0.6, cmap="viridis", interpolation="nearest")
            ax.add_patch(patches.Rectangle((-0.5, -0.5), len(s_idx), len(s_idx), fill=False, edgecolor="gold", lw=2))
            ax.set_title(f"{name} p{ph} {l}\nself {len(s_idx)} ({100 * len(s_idx) / max(alive.sum(), 1):.0f}%)", fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"{title}, tau {TAU_PLOT}")
    plt.tight_layout()
    return fig

plot_self_blocks(lambda r: 1, "After phase 1 (both runs: Hindi from scratch, no switch yet)")
plt.savefig(f"{OUT}/fig_self_block_phase1_tau{TAU_PLOT}.png", dpi=120); plt.show()
plot_self_blocks(lambda r: r["last_phase"], "Latest phase")
plt.savefig(f"{OUT}/fig_self_block_latest_tau{TAU_PLOT}.png", dpi=120); plt.show()

## 7. Over phases: self size, and whether the self keeps the same members

In [ ]:
fig, axes = plt.subplots(2, len(results), figsize=(6.5 * len(results), 7), squeeze=False)
for col, (name, r) in enumerate(results.items()):
    B = r["blocks"][(r["blocks"].tau == TAU_PLOT) & (r["blocks"].phase >= 1)]
    T = r["trans"][r["trans"].tau == TAU_PLOT]
    for l in LAYERS:
        b, t = B[B.layer == l], T[T.layer == l]
        axes[0, col].plot(b.phase, b.self_frac, marker="o", label=l)
        axes[1, col].plot(t.to_phase, t.self_overlap, marker="o", label=l)
    axes[0, col].set_ylim(0, 1); axes[0, col].set_title(f"{name}: self size (fraction of layer)")
    axes[1, col].set_ylim(0, 1); axes[1, col].set_title(f"{name}: same self members as previous phase (Jaccard)")
    for ax in axes[:, col]:
        ax.set_xlabel("phase")
axes[0, 0].legend(ncol=2, fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_self_size_overlap_tau{TAU_PLOT}.png", dpi=120); plt.show()

## 8. Change at each transition: self vs task vs random (lower = more stable)

In [ ]:
ymax = max(r["trans"].task_change.max() for r in results.values() if not r["trans"].empty)
for name, r in results.items():
    T = r["trans"][r["trans"].tau == TAU_PLOT]
    if T.empty:
        print(f"{name}: needs at least 2 trained checkpoints"); continue
    fig, axes = plt.subplots(2, len(LAYERS) // 2, figsize=(4 * len(LAYERS) // 2, 7), sharey=True, squeeze=False)
    for ax, l in zip(axes.flat, LAYERS):
        t = T[T.layer == l]
        ax.plot(t.to_phase, t.self_change, marker="o", label="self")
        ax.plot(t.to_phase, t.task_change, marker="o", label="task")
        ax.plot(t.to_phase, t.random_change, ls="--", label="random (same size)")
        ax.set_title(l); ax.set_xlabel("transition into phase"); ax.set_ylim(0, ymax * 1.05)
    axes[0, 0].set_ylabel("% change"); axes[1, 0].set_ylabel("% change"); axes[0, 0].legend(fontsize=8)
    fig.suptitle(f"{name}: change at each transition, tau {TAU_PLOT}")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_change_{name}_tau{TAU_PLOT}.png", dpi=120); plt.show()

## 9. Summary tables
`gap` = task change − self change (> 0: self more stable). `self_overlap` = how much the self keeps the same neurons from one phase to the next.

In [ ]:
rows = []
for name, r in results.items():
    T = r["trans"]
    if T.empty:
        continue
    for tau in TAUS:
        for l in LAYERS:
            t = T[(T.tau == tau) & (T.layer == l)]
            b = r["blocks"][(r["blocks"].tau == tau) & (r["blocks"].layer == l) & (r["blocks"].phase >= 1)]
            rows.append({"run": name, "tau": tau, "layer": l, "self_frac": b.self_frac.mean(),
                         "self_change": t.self_change.mean(), "task_change": t.task_change.mean(),
                         "random_change": t.random_change.mean(), "gap": t.gap.mean(),
                         "% transitions self more stable": 100 * (t.gap > 0).mean(),
                         "self_overlap": t.self_overlap.mean(), "n_transitions": len(t)})
S = pd.DataFrame(rows)
S.to_csv(f"{OUT}/summary_all_taus.csv", index=False)
print(f"per layer, tau {TAU_PLOT}:")
display(S[S.tau == TAU_PLOT].drop(columns="tau").round(2))
print("averaged over layers, every tau:")
display(S.groupby(["tau", "run"])[["self_frac", "gap", "% transitions self more stable", "self_overlap"]].mean()
         .unstack("run").round(2))
print(f"saved to {OUT}")

## 10. Maps figure (like Figure 18 in the paper)
Average |r| map over the later phases, all in one fixed neuron order, with each neuron's persistence underneath.
Top row: Hindi only. Bottom row: continual. Neurons keep their identity across phases (same weights, trained further), so one order works for every phase.
Order: neurons most often in the self first (gold bar), then the rest. The persistence strip uses connectivity similarity of each neuron between consecutive phases (no matching), averaged over the transitions shown.

In [ ]:
MAP_LAYERS = ["enc1", "enc3", "dec1", "dec3"]   # 4 columns; change if you like
MAP_PHASES = list(range(10, 19))                # later phases, after the first cycles have settled

def map_panel(r, layer):
    phs = [p for p in MAP_PHASES if (p, layer) in r["R"]]
    Rs = [r["R"][(p, layer)][0].astype(np.float32) for p in phs]
    alive = np.logical_and.reduce([r["R"][(p, layer)][1] for p in phs])
    freq = np.mean([self_mask_np(R, alive, TAU_PLOT) for R in Rs], 0)      # how often each neuron is in the self
    conn = []
    for Ra, Rb in zip(Rs[:-1], Rs[1:]):
        A, B = Ra.copy(), Rb.copy(); np.fill_diagonal(A, 0); np.fill_diagonal(B, 0)
        conn.append((A * B).sum(1) / (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12))
    pers = np.mean(conn, 0)
    idx = np.where(alive)[0]
    strength = np.mean([np.abs(R[np.ix_(idx, idx)]).mean(1) for R in Rs], 0)
    order = idx[np.lexsort((-strength, -freq[idx]))]                         # self-frequency first, then strength
    M = np.mean([np.abs(R[np.ix_(order, order)]) for R in Rs], 0)
    return M, pers[order], freq[order], phs

names = ["only_hi", "continual"]
fig = plt.figure(figsize=(4.2 * len(MAP_LAYERS), 11.5))
gs = fig.add_gridspec(2 * len(names), len(MAP_LAYERS), height_ratios=[1, 0.13] * len(names), hspace=0.3, wspace=0.08)
for row, name in enumerate(names):
    for col, layer in enumerate(MAP_LAYERS):
        M, pers, freq, phs = map_panel(results[name], layer)
        n_self = int((freq >= 0.5).sum())
        ax = fig.add_subplot(gs[2 * row, col])
        ax.imshow(M, vmin=0, vmax=0.5, cmap="magma", interpolation="nearest")
        ax.add_patch(patches.Rectangle((-0.5, -0.5), n_self, n_self, fill=False, edgecolor="gold", lw=1.5))
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{name}  {layer}\nself in >=half of phases: {n_self} ({100 * n_self / len(freq):.0f}%)", fontsize=9)
        ax2 = fig.add_subplot(gs[2 * row + 1, col])
        ax2.bar(np.arange(len(pers)), pers, width=1.0, color=np.where(freq >= 0.5, "goldenrod", "grey"))
        ax2.set_xlim(-0.5, len(pers) - 0.5); ax2.set_ylim(0, 1); ax2.set_xticks([])
        if col == 0:
            ax2.set_ylabel("persistence", fontsize=8)
        else:
            ax2.set_yticks([])
fig.suptitle(f"Average |r| over phases {MAP_PHASES[0]}-{MAP_PHASES[-1]}, fixed neuron order, tau {TAU_PLOT}  (top: Hindi only, bottom: continual)")
plt.savefig(f"{OUT}/fig_maps_tau{TAU_PLOT}.png", dpi=130, bbox_inches="tight"); plt.show()

## 11. Paper-style figure (Figures 3 and 19-21 in the paper): one snapshot, one image per layer
Top: co-activation matrix |r| at one checkpoint, neurons reordered so each group (|r| >= tau) is a diagonal block (largest group first, Reverse Cuthill-McKee inside each); the self is outlined in purple, other groups of 10+ neurons in orange.
Bottom: each neuron's persistence at the switch into that checkpoint (same matching and score as section 3), in the same order; dashed lines = mean for self and task.
Left: Hindi only. Right: continual. Needs only sections 1-3 to have run (not the full analysis). Images go to analysis/v4b/paper_style/.

In [ ]:
from scipy.sparse.csgraph import reverse_cuthill_mckee

PAPER_TAUS = [TAU_PLOT]     # e.g. [0.2, 0.3] for one set of images per threshold
PAPER_PHASES = [18]          # snapshot(s) to draw; each uses the switch into that phase (17 -> 18). e.g. [16, 17, 18] = one of each switch type
PAPER_LAYERS = ["enc0", "enc1", "enc2", "enc3", "dec0", "dec1", "dec2", "dec3"]   # one image per layer
MIN_BOX = 10                 # outline groups with at least this many neurons
SELF_COL, TASK_COL = "#4B1D6B", "#B8B8B8"
PDIR = f"{OUT}/paper_style"; os.makedirs(PDIR, exist_ok=True)

def snapshot(path):
    """Traces and correlations of every layer for one checkpoint (same inputs and positions as section 3)."""
    model, ck = load_model(path)
    acts, dec_lang = record(model)
    ie, idd = subsample_idx(acts["enc0"].shape[0], MAX_T), dec_subsample(dec_lang, MAX_T)
    out = {}
    for l, X in acts.items():
        enc = l.startswith("enc")
        Z, alive = traces(X[ie if enc else idd].float(), None if enc else dec_lang[idd])
        out[l] = (Z, Z.T @ Z / Z.shape[0], alive)
    lang = ck["lang"]
    del model, acts
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return out, lang

def paper_panel(prev, cur, layer, tau):
    """Order = groups (largest first, each in Reverse Cuthill-McKee order), then the rest.
    Persistence of each neuron at the switch prev -> cur, as in section 3 (matching, then mean of activation and connectivity similarity)."""
    Za, Ra, aa = prev[layer]; Zb, Rb, ab = cur[layer]
    perm, change, valid = transition(Za, Ra, aa, Zb, Rb, ab)
    perm, change, valid = perm.cpu().numpy(), change.cpu().numpy(), valid.cpu().numpy()
    n = len(perm)
    pers = np.full(n, np.nan); pers[perm[valid]] = 1 - change[valid] / 100     # indexed by the neuron at the later snapshot
    R, alive = Rb.cpu().numpy(), ab.cpu().numpy()
    A = (np.abs(R) >= tau) & alive[:, None] & alive[None, :]
    np.fill_diagonal(A, False)
    _, lab = connected_components(csr_matrix(A), directed=False)
    lab = np.where(alive, lab, -1)
    sizes = np.bincount(lab[lab >= 0])
    groups = [g for g in np.argsort(-sizes) if sizes[g] >= 2]
    order, boxes = [], []
    for k, g in enumerate(groups):
        idx = np.where(lab == g)[0]
        sub = reverse_cuthill_mckee(csr_matrix(A[np.ix_(idx, idx)]), symmetric_mode=True)
        boxes.append((len(order), len(idx), k == 0))
        order.extend(idx[sub])
    used = set(order)
    rest = [i for i in range(n) if i not in used]
    order = np.array(order + rest)
    is_self = np.zeros(n, bool); is_self[np.where(lab == groups[0])[0]] = True
    return np.abs(R[np.ix_(order, order)]), boxes, pers[order], is_self[order]

def paper_figure(panels, layer, phase, langs, tau):
    fig = plt.figure(figsize=(12, 7.6))
    gs = fig.add_gridspec(2, len(panels), height_ratios=[1, 0.28], hspace=0.08, wspace=0.12)
    for col, (name, (M, boxes, pers, is_self)) in enumerate(panels.items()):
        ax = fig.add_subplot(gs[0, col])
        ax.imshow(M, cmap="Greys", vmin=0, vmax=0.6, interpolation="nearest")
        for start, size, self_box in boxes:
            if size >= MIN_BOX or self_box:
                ax.add_patch(patches.Rectangle((start - 0.5, start - 0.5), size, size, fill=False,
                                               edgecolor=SELF_COL if self_box else "#E07A1F", lw=2.2 if self_box else 0.8))
        n_self = int(is_self.sum())
        ax.set_title(f"{name}: phase {phase - 1} → {phase} ({langs[name]})\nself {n_self} of {len(is_self)} neurons ({100 * n_self / len(is_self):.0f}%)", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        ax2 = fig.add_subplot(gs[1, col])
        x = np.arange(len(pers))
        ax2.bar(x, np.nan_to_num(pers), width=1.0, color=np.where(is_self, SELF_COL, TASK_COL))
        ms, mt = np.nanmean(pers[is_self]), np.nanmean(pers[~is_self])
        ax2.axhline(ms, color=SELF_COL, ls="--", lw=1); ax2.axhline(mt, color="#555555", ls="--", lw=1)
        ax2.text(len(pers) * 0.99, 0.04, f"mean persistence: self {ms:.2f} · task {mt:.2f}", ha="right", va="bottom", fontsize=8,
                 bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))
        ax2.set_xlim(-0.5, len(pers) - 0.5); ax2.set_ylim(0, 1); ax2.set_xticks([])
        ax2.set_xlabel("neurons, same order as above", fontsize=8)
        if col == 0:
            ax2.set_ylabel("persistence", fontsize=9)
        else:
            ax2.set_yticks([])
    fig.suptitle(f"Layer {layer}: co-activation |r| (reordered, groups at |r| ≥ {tau} outlined, self in purple) and per-neuron persistence", fontsize=11)
    return fig

names = ["only_hi", "continual"]
for phase in PAPER_PHASES:
    snaps, langs = {}, {}
    for name in names:
        by_phase = {phase_of(p): p for p in avail[name]}
        prev, la = snapshot(by_phase[phase - 1])
        cur, lb = snapshot(by_phase[phase])
        langs[name] = f"{la} → {lb}" if la != lb else lb
        snaps[name] = {(l, t): paper_panel(prev, cur, l, t) for l in PAPER_LAYERS for t in PAPER_TAUS}
        del prev, cur
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    for layer in PAPER_LAYERS:
      for tau in PAPER_TAUS:
        fig = paper_figure({nm: snaps[nm][(layer, tau)] for nm in names}, layer, phase, langs, tau)
        fig.savefig(f"{PDIR}/{layer}_p{phase:02d}_tau{tau}.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close(fig)
print("saved to", PDIR)


## 12. 2x2: where a neuron came from vs where it is now
At every switch, each neuron is matched to its partner in the next checkpoint (as in section 3).
Rows: was it in the self or the task group before the switch. Columns: is its partner in the self or the task group after.
Entry: mean persistence (neuron similarity = mean of activation and connectivity similarity, 1 = identical), pooled over all switches and layers.
Small text: share of the row (e.g. self → self = how much of the self stays in the self).

In [ ]:
def quad_table(Q, tau=TAU_PLOT, layer=None):
    q = Q[(Q.tau == tau) & (Q.n > 0)]
    if layer is not None:
        q = q[q.layer == layer]
    q = q.assign(w=q.persistence * q.n).groupby(["came_from", "now_in"])[["w", "n"]].sum()
    pers = (q.w / q.n).unstack().loc[["self", "task"], ["self", "task"]]
    share = q.n.unstack().loc[["self", "task"], ["self", "task"]]
    share = share.div(share.sum(axis=1), axis=0)
    return pers, share

names = ["only_hi", "continual"]
QUAD_TAUS = [0.1, 0.15, 0.2, 0.25, 0.3]          # one row of heatmaps per tau
fig, axes = plt.subplots(len(QUAD_TAUS), len(names), figsize=(4.6 * len(names), 3.9 * len(QUAD_TAUS)), squeeze=False)
for row, tau in enumerate(QUAD_TAUS):
    for ax, name in zip(axes[row], names):
        pers, share = quad_table(results[name]["quad"], tau=tau)
        ax.imshow(pers.values, cmap="Purples", vmin=0.3, vmax=1.0)
        for a in range(2):
            for b in range(2):
                v = pers.values[a, b]
                col = "white" if v > 0.75 else "black"
                ax.text(b, a - 0.08, f"{v:.2f}", ha="center", va="center", fontsize=18, fontweight="bold", color=col)
                ax.text(b, a + 0.24, f"{100 * share.values[a, b]:.0f}% of row", ha="center", va="center", fontsize=9, color=col)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["self", "task"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["self", "task"])
        ax.set_xlabel("is now in (after the switch)", fontsize=9); ax.set_ylabel("came from (before)", fontsize=9)
        sz = results[name]["blocks"]; sz = sz[(sz.tau == tau) & (sz.phase >= 1)].self_frac.mean()
        ax.set_title(f"{name}, tau {tau}  (self = {100 * sz:.0f}% of layer)", fontsize=10)
fig.suptitle("Mean persistence by where a neuron came from and where it is now (all switches, all layers)", y=1.0)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_quad_taus.png", dpi=130, bbox_inches="tight"); plt.show()

rows = []
for name in names:
    for l in LAYERS:
        pers, share = quad_table(results[name]["quad"], layer=l)
        rows.append({"run": name, "layer": l,
                     **{f"{a}->{b}": round(pers.loc[a, b], 3) for a in ["self", "task"] for b in ["self", "task"]},
                     **{f"share {a}->{b}": round(share.loc[a, b], 2) for a in ["self", "task"] for b in ["self", "task"]}})
QT = pd.DataFrame(rows); QT.to_csv(f"{OUT}/quad_per_layer_tau{TAU_PLOT}.csv", index=False)
print("per layer (persistence, then share of row):"); display(QT)